In [2]:
import requests

TOKEN = "8700714411:AAFSb3k9yKBMJWuBZh24lPVHtjjqMyQg6O0"

url = f"https://api.telegram.org/bot{TOKEN}/getUpdates"
print(requests.get(url).json())

{'ok': True, 'result': [{'update_id': 866949121, 'message': {'message_id': 3, 'from': {'id': 1193239390, 'is_bot': False, 'first_name': 'Arvind', 'last_name': 'Kumar', 'language_code': 'en'}, 'chat': {'id': 1193239390, 'first_name': 'Arvind', 'last_name': 'Kumar', 'type': 'private'}, 'date': 1773938210, 'text': 'hi'}}]}


In [ ]:
import MetaTrader5 as mt5
import pandas as pd
from datetime import datetime, timedelta
import pytz
import time


import requests

BOT_TOKEN = "8700714411:AAFSb3k9yKBMJWuBZh24lPVHtjjqMyQg6O0"
CHAT_ID = "1193239390"

def send_telegram(msg):
    try:
        url = f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage"
        requests.post(url, data={
            "chat_id": CHAT_ID,
            "text": msg
        })
    except Exception as e:
        print("Telegram Error:", e)
        
import builtins

old_print = print

def new_print(*args, **kwargs):
    message = " ".join(str(a) for a in args)
    old_print(*args, **kwargs)
    send_telegram(message)

builtins.print = new_print


# ==============================
# SETTINGS
# ==============================

MODE = "LIVE"  # BACKTEST or LIVE
DEBUG = True
SYMBOL = "XAUUSDm"
TIMEFRAME = mt5.TIMEFRAME_M15
LOT_SIZE = 0.01
MAGIC_NUMBER = 108108
BACKTEST_DAYS = 365
MINTP=2.75
MAXTP=9
MAXTRADEPERSESSION=5
TPBEFOREPIVOTPOINTS=2
SPREAD_SIM = 0.30


# ==============================
# MT5 INIT
# ==============================

if not mt5.initialize():
    print("MT5 initialization failed")
    quit()


# ==============================
# GLOBAL STATE
# ==============================

pivot_taps = {
"P":{"buy":0,"sell":0},
"R1":{"buy":0,"sell":0},
"R2":{"buy":0,"sell":0},
"R3":{"buy":0,"sell":0},
"S1":{"buy":0,"sell":0},
"S2":{"buy":0,"sell":0},
"S3":{"buy":0,"sell":0}
}

pivot_first_tap_time = {
"P":{"buy":None,"sell":None},
"R1":{"buy":None,"sell":None},
"R2":{"buy":None,"sell":None},
"R3":{"buy":None,"sell":None},
"S1":{"buy":None,"sell":None},
"S2":{"buy":None,"sell":None},
"S3":{"buy":None,"sell":None}
}

session_trade_count = {"Asian":0,"London":0,"NewYork":0}


trade_active = False


# ==============================
# SEASON DETECTION
# ==============================

def get_season(dt):

    if dt.month in [4,5,6,7,8,9]:
        return "Summer"
    else:
        return "Winter"


# ==============================
# TRADING TIME FILTER
# ==============================

def trading_allowed(dt):

    season = get_season(dt)
    hour = dt.hour
    weekday = dt.weekday()

    if season == "Winter":
        if hour in  [1,3,5,6,8,12,15,16,17,18,19,20,21,22,23]: #[1,3,4,5,6,8,10,12,13,16,17,18,19,20,21,22,23] : 
            return False

    if season == "Summer":
        if hour in [1,2,3,4,5,6,7,8,9,10,11,15,16,17,18,20]: #[0,1,2,3,4,5,6,7,8,10,12,15,16,17,18,20]:
            return False
        
    if weekday == 0 and hour in [4,10]: # [15]:
        return False
    if weekday == 1 and hour in [2,10]: # [14]:
        return False
    if weekday == 2 and hour in [0,14]: #[13]:
        return False
    if weekday == 3 and hour in [0,15]: #[13]:
        return False
    if weekday == 4 and hour in [0,9]: #[14]:
        return False
    
    

    return True

def is_trade_active():

    positions = mt5.positions_get()

    if positions is None:
        return False

    for pos in positions:

        if pos.symbol == SYMBOL and pos.magic == MAGIC_NUMBER:
            return True

    return False


# ==============================
# SESSION DETECTION
# ==============================

def get_session(dt):

    h = dt.hour

    if 0 <= h < 8:
        return "Asian"
    elif 8 <= h < 16:
        return "London"
    else:
        return "NewYork"


# ==============================
# FIBONACCI PIVOTS
# ==============================

def calculate_fibonacci_pivots():

    rates = mt5.copy_rates_from_pos(SYMBOL, mt5.TIMEFRAME_M15, 0, 60000)
    df = pd.DataFrame(rates)

    df["time"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df["time_ny"] = df["time"].dt.tz_convert("America/New_York")

    df["session_date"] = (df["time_ny"] - pd.Timedelta(hours=17)).dt.date

    daily = df.groupby("session_date").agg({
        "high":"max",
        "low":"min",
        "close":"last"
    }).reset_index()

    H = daily["high"].shift(1)
    L = daily["low"].shift(1)
    C = daily["close"].shift(1)

    P = (H + L + C) / 3
    range_ = H - L

    daily["P"] = P
    daily["R1"] = P + 0.382 * range_
    daily["R2"] = P + 0.618 * range_
    daily["R3"] = P + 1.000 * range_

    daily["S1"] = P - 0.382 * range_
    daily["S2"] = P - 0.618 * range_
    daily["S3"] = P - 1.000 * range_

    for col in ["P","R1","R2","R3","S1","S2","S3"]:
        daily[col] = daily[col].round(4)

    daily.rename(columns={"session_date":"date"}, inplace=True)

    return daily[["date","P","R1","R2","R3","S1","S2","S3"]]


# ==============================
# TAP DETECTION
# ==============================

def detect_tap(candle, level):

    o = candle["open"]
    c = candle["close"]
    l = candle["low"]
    h = candle["high"]

    body = abs(c - o)
    range_ = h - l

    if range_ == 0:
        return False, False

    body_ratio = body / range_

    if DEBUG and MODE == "LIVE":
        print("Body ratio:", round(body_ratio,3))

    if body_ratio < 0.35:
        if DEBUG and MODE == "LIVE":
            print("Rejected: Weak candle")
        return False, False

    buy_tap = l < level and c > level and c > o
    sell_tap = o > level and c < level and c < o

    if DEBUG:
        print("Buy Tap:", buy_tap, "Sell Tap:", sell_tap)

    return buy_tap, sell_tap

def debug_candle(candle, pivots):

    if not DEBUG  or MODE != "LIVE":
        return

    print("\n================ NEW CLOSED CANDLE ================")

    print("Time:", candle["time"])
    print("Open:", candle["open"],
          "High:", candle["high"],
          "Low:", candle["low"],
          "Close:", candle["close"])

    body = abs(candle["close"] - candle["open"])
    rng = candle["high"] - candle["low"]

    body_ratio = 0 if rng == 0 else round(body / rng, 3)

    print("Body Ratio:", body_ratio)

    print("\nPivot Levels:")
    for p in ["P","R1","R2","R3","S1","S2","S3"]:
        print(p, ":", pivots[p])

    print("===================================================")

# ==============================
# TARGET MAP
# ==============================

target_buy = {
"S3":"S2","S2":"S1","S1":"P","P":"R1","R1":"R2","R2":"R3","R3":None
}

target_sell = {
"S3":None,"S2":"S3","S1":"S2","P":"S1","R1":"P","R2":"R1","R3":"R2"
}

# ==============================
# PERFORMANCE SUMMARY
# ==============================

def performance_summary(trades_df):

    if trades_df.empty:
        print("No trades generated.")
        return

    trades_df["EntryTime"] = pd.to_datetime(trades_df["EntryTime"])

    trades_df["Month"] = trades_df["EntryTime"].dt.strftime("%b-%y")

    months = trades_df["Month"].unique()[-12:]

    print("\n===============================================")
    print("BACKTEST PERFORMANCE SUMMARY (LAST 12 MONTHS)")
    print("===============================================\n")

    equity = 0
    equity_curve = []

    for m in months:

        mdf = trades_df[trades_df["Month"] == m]

        total = len(mdf)
        wins = len(mdf[mdf["PnL"] > 0])
        losses = len(mdf[mdf["PnL"] <= 0])

        pnl = mdf["PnL"].sum()

        acc = round((wins / total) * 100, 2) if total > 0 else 0

        print(
            f"Month: {m} | Total Trades: {total} | Winning Trades: {wins} | "
            f"Losing Trades: {losses} | PnL: {round(pnl,2)} | Accuracy: {acc}%"
        )

        equity += pnl
        equity_curve.append(equity)

    print("\n-----------------------------------------------")

    total_trades = len(trades_df)
    total_wins = len(trades_df[trades_df["PnL"] > 0])
    total_losses = len(trades_df[trades_df["PnL"] <= 0])

    total_pnl = trades_df["PnL"].sum()

    accuracy = round((total_wins / total_trades) * 100,2)

    # ==============================
    # MAX DRAWDOWN
    # ==============================

    # ==========================================
    # REAL EQUITY CURVE DRAWDOWN (TRADE BASED)
    # ==========================================

    trades_df = trades_df.sort_values("EntryTime")

    equity = 0
    peak = 0
    max_dd = 0

    equity_curve = []

    for pnl in trades_df["PnL"]:

        equity += pnl
        equity_curve.append(equity)

        if equity > peak:
            peak = equity

        dd = equity - peak

        if dd < max_dd:
            max_dd = dd
            
            
        # ==========================================
    # HOUR WISE PERFORMANCE (SEASON WISE)
    # ==========================================

    print("\n============= HOUR WISE PERFORMANCE =============\n")

    for season in ["Winter", "Summer"]:

        sdf = trades_df[trades_df["Season"] == season]

        if sdf.empty:
            continue

        hh_stats = sdf.groupby("HH").agg(
            Trades=("PnL", "count"),
            Wins=("PnL", lambda x: (x > 0).sum()),
            Losses=("PnL", lambda x: (x <= 0).sum()),
            PnL=("PnL", "sum")
        ).reset_index()

        hh_stats["Accuracy"] = round(
            (hh_stats["Wins"] / hh_stats["Trades"]) * 100, 2
        )

        print(f"\n------ {season} Season ------\n")

        for _, r in hh_stats.iterrows():

            print(
                f"HH: {int(r['HH']):02d} | "
                f"Trades: {int(r['Trades'])} | "
                f"Wins: {int(r['Wins'])} | "
                f"Losses: {int(r['Losses'])} | "
                f"PnL: {round(r['PnL'],2)} | "
                f"Accuracy: {r['Accuracy']}%"
            )

    print("\n============== OVERALL SUMMARY ==============")

    print("Total Trades :", total_trades)
    print("Winning Trades :", total_wins)
    print("Losing Trades :", total_losses)
    print("Accuracy :", accuracy,"%")
    print("Total PnL :", round(total_pnl,2))
    print("Max Drawdown :", round(max_dd,2))

    print("=============================================\n")


# ==============================
# ORDER EXECUTION
# ==============================

def place_order(side, entry, sl, tp):

    tick = mt5.symbol_info_tick(SYMBOL)

    if tick is None:
        print("Tick data unavailable")
        return

    ask = tick.ask
    bid = tick.bid

    expiration = datetime.now() + timedelta(minutes=30)

    # ==================================
    # BUY LOGIC
    # ==================================

    if side == "BUY":

        if entry >= ask:
            # MARKET BUY
            request = {
                "action": mt5.TRADE_ACTION_DEAL,
                "symbol": SYMBOL,
                "volume": LOT_SIZE,
                "type": mt5.ORDER_TYPE_BUY,
                "price": ask,
                "sl": sl,
                "tp": tp,
                "magic": MAGIC_NUMBER,
                "comment": "PivotDoubleTap",
                "type_time": mt5.ORDER_TIME_GTC,
                "type_filling": mt5.ORDER_FILLING_IOC,
            }

        else:
            # BUY LIMIT
            request = {
                "action": mt5.TRADE_ACTION_PENDING,
                "symbol": SYMBOL,
                "volume": LOT_SIZE,
                "type": mt5.ORDER_TYPE_BUY_LIMIT,
                "price": entry,
                "sl": sl,
                "tp": tp,
                "magic": MAGIC_NUMBER,
                "comment": "PivotDoubleTap",
                "type_time": mt5.ORDER_TIME_SPECIFIED,
                "expiration": expiration,
                "type_filling": mt5.ORDER_FILLING_RETURN,
            }

    # ==================================
    # SELL LOGIC
    # ==================================

    else:

        if entry <= bid:
            # MARKET SELL
            request = {
                "action": mt5.TRADE_ACTION_DEAL,
                "symbol": SYMBOL,
                "volume": LOT_SIZE,
                "type": mt5.ORDER_TYPE_SELL,
                "price": bid,
                "sl": sl,
                "tp": tp,
                "magic": MAGIC_NUMBER,
                "comment": "PivotDoubleTap",
                "type_time": mt5.ORDER_TIME_GTC,
                "type_filling": mt5.ORDER_FILLING_IOC,
            }

        else:
            # SELL LIMIT
            request = {
                "action": mt5.TRADE_ACTION_PENDING,
                "symbol": SYMBOL,
                "volume": LOT_SIZE,
                "type": mt5.ORDER_TYPE_SELL_LIMIT,
                "price": entry,
                "sl": sl,
                "tp": tp,
                "magic": MAGIC_NUMBER,
                "comment": "PivotDoubleTap",
                "type_time": mt5.ORDER_TIME_SPECIFIED,
                "expiration": expiration,
                "type_filling": mt5.ORDER_FILLING_RETURN,
            }

    result = mt5.order_send(request)

    print("Order Result:", result)

# ==============================
# STRATEGY ENGINE
# ==============================

def process_candle(candle, pivots):

    global pivot_taps
    global session_trade_count
    global trade_active
    global pivot_first_tap_time

    if MODE == "BACKTEST" and trade_active:
        return None

    if MODE == "LIVE" and is_trade_active():
        
        if DEBUG and MODE == "LIVE":
            print("Blocked: Trade already active")
        return None

    if not trading_allowed(candle["time"]):
        if DEBUG:
            print("Blocked: Trading time filter")
        return None

    session = get_session(candle["time"])

    if session_trade_count[session] >= MAXTRADEPERSESSION:
        if DEBUG:
            print("Blocked: Max trades reached in session")
        return None

    for pivot in pivot_taps.keys():

        level = pivots[pivot]

        buy_tap, sell_tap = detect_tap(candle, level)
        
        if DEBUG and MODE == "LIVE":
            print("\nChecking Pivot:", pivot, "Level:", pivots[pivot])

        if buy_tap:

            pivot_taps[pivot]["buy"] += 1

            if pivot_first_tap_time[pivot]["buy"] is None:
                pivot_first_tap_time[pivot]["buy"] = candle["time"]

            if pivot_taps[pivot]["buy"] >= 2:

                close = candle["close"]
                low = candle["low"]

                range_ = close - low

                if range_ > 10:
                    entry = low + range_/2
                else:
                    entry = close

                sl = candle["low"] - 2
                risk = entry - sl

                next_pivot = target_buy[pivot]

                min_tp = entry + MINTP*risk
                max_tp = entry + MAXTP*risk

                if next_pivot is None:
                    tp = min_tp
                else:
                    pivot_tp = pivots[next_pivot] - TPBEFOREPIVOTPOINTS
                    tp = max(pivot_tp, min_tp)

                # CAP TP at 1:5 RR
                if tp > max_tp:
                    tp = max_tp

                session_trade_count[session] += 1

                return ("BUY", entry, sl, tp, pivot)

        if sell_tap:

            pivot_taps[pivot]["sell"] += 1

            if pivot_first_tap_time[pivot]["sell"] is None:
                pivot_first_tap_time[pivot]["sell"] = candle["time"]

            if pivot_taps[pivot]["sell"] >= 2:

                close = candle["close"]
                high = candle["high"]

                range_ = high - close

                if range_ > 10:
                    entry = high - range_/2
                else:
                    entry = close

                sl = candle["high"] + 2
                risk = sl - entry

                next_pivot = target_sell[pivot]

                min_tp = entry - MINTP*risk
                max_tp = entry - MAXTP*risk

                if next_pivot is None:
                    tp = min_tp
                else:
                    pivot_tp = pivots[next_pivot] + TPBEFOREPIVOTPOINTS
                    tp = min(pivot_tp, min_tp)

                # CAP TP at 1:5 RR
                if tp < max_tp:
                    tp = max_tp

                session_trade_count[session] += 1

                return ("SELL", entry, sl, tp, pivot)

            #return None


# ==============================
# BACKTEST ENGINE
# ==============================

def run_backtest():

    pivots_df = calculate_fibonacci_pivots()

    end = datetime.utcnow()
    start = end - timedelta(days=BACKTEST_DAYS)

    rates = mt5.copy_rates_range(SYMBOL, TIMEFRAME, start, end)
    df = pd.DataFrame(rates)

    df["time"] = pd.to_datetime(df["time"], unit="s")

    trades = []
    current_day = None

    i = 0

    while i < len(df):

        candle = df.iloc[i]

        ny = pytz.timezone("America/New_York")

        candle_time_utc = candle["time"].tz_localize("UTC")
        candle_ny = candle_time_utc.astimezone(ny)

        day = (candle_ny - pd.Timedelta(hours=17)).date()

        # =========================
        # NEW TRADING DAY
        # =========================
        if day != current_day:

            row = pivots_df[pivots_df["date"] == day]

            if row.empty:
                i += 1
                continue

            pivots = row.iloc[0]
            debug_candle(candle, pivots)

            for k in pivot_taps:
                pivot_taps[k]["buy"] = 0
                pivot_taps[k]["sell"] = 0

            for k in pivot_first_tap_time:
                pivot_first_tap_time[k]["buy"] = None
                pivot_first_tap_time[k]["sell"] = None

            for k in session_trade_count:
                session_trade_count[k] = 0

            current_day = day

        # =========================
        # CHECK FOR SIGNAL
        # =========================
        signal = process_candle(candle, pivots)

        if signal is None:
            i += 1
            continue

        side, entry, sl, tp, pivot = signal

        signal_time = candle["time"]
        entry_time = None
        
        # =========================
        # CANDLE TYPE
        # =========================

        if candle["close"] > candle["open"]:
            candle_type = "Bullish"
        else:
            candle_type = "Bearish"
        

        # =========================
        # ORDER TYPE DETECTION
        # =========================

        if side == "BUY":
            order_type = "Market" if entry >= candle["close"] else "Pending"
        else:
            order_type = "Market" if entry <= candle["close"] else "Pending"

        trade_closed = False

        # =========================
        # MARKET ORDER EXECUTION
        # =========================
        if order_type == "Market":

            entry = candle["close"]
            entry_time = signal_time
            entry_hit = True

        else:
            entry_hit = False
            
        # scan future candle:
        

        # =========================
        # ENTRY WINDOW (30 MIN = 2 CANDLES)
        # =========================

        #entry_hit = False

        # MARKET ORDER
        if order_type == "Market":

            if side == "BUY":
                entry = candle["close"] + SPREAD_SIM
            else:
                entry = candle["close"] - SPREAD_SIM

            entry_time = signal_time
            entry_hit = True
            entry_index = i

        # PENDING ORDER
        else:

            for j in range(i+1, min(i+3, len(df))):

                future = df.iloc[j]

                if side == "BUY" and future["low"] <= entry:
                    entry_hit = True
                    entry_time = future["time"]
                    entry_index = j
                    break

                if side == "SELL" and future["high"] >= entry:
                    entry_hit = True
                    entry_time = future["time"]
                    entry_index = j
                    break

        # cancel if entry not triggered
        if not entry_hit:
            i += 1
            continue
            
        # =========================
        # EXIT TRACKING (NO TIME LIMIT)
        # =========================

        for j in range(entry_index+1, len(df)):

            future = df.iloc[j]

            if side == "BUY":

                if future["low"] <= sl:
                    pnl = sl - entry
                    exit_price = sl
                    exit_time = future["time"]
                    trade_closed = True
                    break

                if future["high"] >= tp:
                    pnl = tp - entry
                    exit_price = tp
                    exit_time = future["time"]
                    trade_closed = True
                    break

            else:

                if future["high"] >= sl:
                    pnl = entry - sl
                    exit_price = sl
                    exit_time = future["time"]
                    trade_closed = True
                    break

                if future["low"] <= tp:
                    pnl = entry - tp
                    exit_price = tp
                    exit_time = future["time"]
                    trade_closed = True
                    break
    
    
        # =========================
        # RECORD TRADE
        # =========================
        if trade_closed:

            trades.append({
                "SignalCandleTime": signal_time,
                "EntryTime": entry_time,
                "ExitTime": exit_time,
                "Type": side,
                "Pivot": pivot,
                "First tap Time": pivot_first_tap_time[pivot]["buy"] if side=="BUY" else pivot_first_tap_time[pivot]["sell"],
                "OrderType": order_type,
                "CandleType": candle_type,
                "Entry": entry,
                "SL": sl,
                "TP": tp,
                "Exit": exit_price,
                "PnL": pnl,

                "P": pivots["P"],
                "R1": pivots["R1"],
                "R2": pivots["R2"],
                "R3": pivots["R3"],
                "S1": pivots["S1"],
                "S2": pivots["S2"],
                "S3": pivots["S3"],

                "HH": entry_time.hour,
                "Month": entry_time.strftime("%b"),
                "Day": entry_time.strftime("%A"),
                "Season": get_season(entry_time),
                "Session": get_session(entry_time)
            })

            # jump to exit candle
            i = j
            continue

        # if trade never closed (rare)
        i += 1

    # =========================
    # SAVE RESULTS
    # =========================
    trades_df = pd.DataFrame(trades)

    if trades_df.empty:
        print("No trades generated.")
        return

    trades_df["EntryTime"] = pd.to_datetime(trades_df["EntryTime"])
    trades_df["ExitTime"] = pd.to_datetime(trades_df["ExitTime"])

    trades_df["HH"] = trades_df["EntryTime"].dt.hour
    trades_df["Month"] = trades_df["EntryTime"].dt.strftime("%b")
    trades_df["Day"] = trades_df["EntryTime"].dt.strftime("%A")
    trades_df["Season"] = trades_df["EntryTime"].apply(get_season)
    trades_df["Session"] = trades_df["EntryTime"].apply(get_session)

    trades_df.to_excel("Pivot_Double_tap_Backtest_Trades.xlsx", index=False)

    print("Backtest Completed")
    print("Results saved to Pivot_Double_tap_Backtest_Trades.xlsx")

    performance_summary(trades_df)

# ==============================
# LIVE TRADING LOOP
# ==============================

def live_loop():

    pivots_df = calculate_fibonacci_pivots()

    last_bar = None
    last_pivot_day = None

    while True:

        rates = mt5.copy_rates_from_pos(SYMBOL, TIMEFRAME, 0, 2)
        df = pd.DataFrame(rates)

        candle = df.iloc[-2].to_dict()
        candle["time"] = datetime.utcfromtimestamp(candle["time"])

        if last_bar == candle["time"]:
            time.sleep(2)
            continue

        last_bar = candle["time"]

        # ==========================
        # DAILY PIVOT RESET
        # ==========================

        ny = pytz.timezone("America/New_York")

        candle_time_utc = candle["time"].replace(tzinfo=pytz.utc)
        candle_ny = candle_time_utc.astimezone(ny)

        today = (candle_ny - pd.Timedelta(hours=17)).date()

        if today != last_pivot_day:

            pivots_df = calculate_fibonacci_pivots()

            for k in pivot_taps:
                pivot_taps[k]["buy"] = 0
                pivot_taps[k]["sell"] = 0

            for k in pivot_first_tap_time:
                pivot_first_tap_time[k]["buy"] = None
                pivot_first_tap_time[k]["sell"] = None

            for k in session_trade_count:
                session_trade_count[k] = 0

            last_pivot_day = today

        row = pivots_df[pivots_df["date"] == today]

        if row.empty:
            time.sleep(2)
            continue

        pivots = row.iloc[0]

        # ==========================
        # SPREAD FILTER
        # ==========================

        tick = mt5.symbol_info_tick(SYMBOL)

        if tick is None:
            time.sleep(2)
            continue

        spread = tick.ask - tick.bid

        if spread > 3:
            if DEBUG:
                print("Blocked: Spread too high ->", spread)
            time.sleep(2)
            continue

        # ==========================
        # SIGNAL CHECK
        # ==========================

        signal = process_candle(candle, pivots)

        if signal:

            side, entry, sl, tp, pivot = signal

            if is_trade_active():
                continue

            place_order(side, entry, sl, tp)

            print(
                "LIVE TRADE:",
                side,
                "Pivot:",
                pivot,
                "Entry:",
                entry,
                "SL:",
                sl,
                "TP:",
                tp
            )

        time.sleep(2)
# ==============================
# MAIN
# ==============================

if __name__=="__main__":

    if MODE=="BACKTEST":
        run_backtest()

    if MODE=="LIVE":
        live_loop()

Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Body ratio: 0.431
Buy Tap: True Sell Tap: F


Checking Pivot: R2 Level: 4900.8553
Body ratio: 0.215
Rejected: Weak candle

Checking Pivot: R3 Level: 5040.055
Body ratio: 0.215
Rejected: Weak candle

Checking Pivot: S1 Level: 4536.4583
Body ratio: 0.215
Rejected: Weak candle

Checking Pivot: S2 Level: 4450.4607
Body ratio: 0.215
Rejected: Weak candle

Checking Pivot: S3 Level: 4311.261
Body ratio: 0.235
Rejected: Weak candle

Checking Pivot: P Level: 4675.658
Body ratio: 0.235
Rejected: Weak candle

Checking Pivot: R1 Level: 4814.8577
Body ratio: 0.235
Rejected: Weak candle

Checking Pivot: R2 Level: 4900.8553
Body ratio: 0.235
Rejected: Weak candle

Checking Pivot: R3 Level: 5040.055
Body ratio: 0.235
Rejected: Weak candle

Checking Pivot: S1 Level: 4536.4583
Body ratio: 0.235
Rejected: Weak candle

Checking Pivot: S2 Level: 4450.4607
Body ratio: 0.235
Rejected: Weak candle

Checking Pivot: S3 Level: 4311.261
Body ratio: 0.094
Rejected: Weak candle

Checking Pivot: P Level: 4675.658
Body ratio: 0.094
Rejected: Weak candle

Checki

Body ratio: 0.624
Buy Tap: False Sell Tap: False

Checking Pivot: S1 Level: 4536.4583
Body ratio: 0.624
Buy Tap: False Sell Tap: False

Checking Pivot: S2 Level: 4450.4607
Body ratio: 0.624
Buy Tap: False Sell Tap: False

Checking Pivot: S3 Level: 4311.261
Body ratio: 0.329
Rejected: Weak candle

Checking Pivot: P Level: 4675.658
Body ratio: 0.329
Rejected: Weak candle

Checking Pivot: R1 Level: 4814.8577
Body ratio: 0.329
Rejected: Weak candle

Checking Pivot: R2 Level: 4900.8553
Body ratio: 0.329
Rejected: Weak candle

Checking Pivot: R3 Level: 5040.055
Body ratio: 0.329
Rejected: Weak candle

Checking Pivot: S1 Level: 4536.4583
Body ratio: 0.329
Rejected: Weak candle

Checking Pivot: S2 Level: 4450.4607
Body ratio: 0.329
Rejected: Weak candle

Checking Pivot: S3 Level: 4311.261
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Block


Checking Pivot: R2 Level: 4730.0374
Body ratio: 0.904
Buy Tap: False Sell Tap: False

Checking Pivot: R3 Level: 4828.8833
Body ratio: 0.904
Buy Tap: False Sell Tap: False

Checking Pivot: S1 Level: 4471.2784
Body ratio: 0.904
Buy Tap: False Sell Tap: False

Checking Pivot: S2 Level: 4410.2113
Body ratio: 0.904
Buy Tap: False Sell Tap: False

Checking Pivot: S3 Level: 4311.3653
Body ratio: 0.228
Rejected: Weak candle

Checking Pivot: P Level: 4570.1243
Body ratio: 0.228
Rejected: Weak candle

Checking Pivot: R1 Level: 4668.9703
Body ratio: 0.228
Rejected: Weak candle

Checking Pivot: R2 Level: 4730.0374
Body ratio: 0.228
Rejected: Weak candle

Checking Pivot: R3 Level: 4828.8833
Body ratio: 0.228
Rejected: Weak candle

Checking Pivot: S1 Level: 4471.2784
Body ratio: 0.228
Rejected: Weak candle

Checking Pivot: S2 Level: 4410.2113
Body ratio: 0.228
Rejected: Weak candle

Checking Pivot: S3 Level: 4311.3653
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time f

Body ratio: 0.672
Buy Tap: False Sell Tap: False

Checking Pivot: S2 Level: 4410.2113
Body ratio: 0.672
Buy Tap: False Sell Tap: False

Checking Pivot: S3 Level: 4311.3653
Body ratio: 0.392
Buy Tap: False Sell Tap: False

Checking Pivot: P Level: 4570.1243
Body ratio: 0.392
Buy Tap: False Sell Tap: False

Checking Pivot: R1 Level: 4668.9703
Body ratio: 0.392
Buy Tap: False Sell Tap: False

Checking Pivot: R2 Level: 4730.0374
Body ratio: 0.392
Buy Tap: False Sell Tap: False

Checking Pivot: R3 Level: 4828.8833
Body ratio: 0.392
Buy Tap: False Sell Tap: False

Checking Pivot: S1 Level: 4471.2784
Body ratio: 0.392
Buy Tap: False Sell Tap: False

Checking Pivot: S2 Level: 4410.2113
Body ratio: 0.392
Buy Tap: False Sell Tap: False

Checking Pivot: S3 Level: 4311.3653
Body ratio: 0.423
Buy Tap: False Sell Tap: False

Checking Pivot: P Level: 4570.1243
Body ratio: 0.423
Buy Tap: False Sell Tap: False

Checking Pivot: R1 Level: 4668.9703
Body ratio: 0.423
Buy Tap: False Sell Tap: False

Checki

...

Rejected: Weak candle

Checking Pivot: S1 Level: 4353.9335
Body ratio: 0.036
Rejected: Weak candle

Checking Pivot: S2 Level: 4311.7888
Body ratio: 0.036
Rejected: Weak candle

Checking Pivot: S3 Level: 4243.5717
Body ratio: 0.537
Buy Tap: False Sell Tap: False

Checking Pivot: P Level: 4422.1507
Body ratio: 0.537
Buy Tap: False Sell Tap: False

Checking Pivot: R1 Level: 4490.3678
Body ratio: 0.537
Buy Tap: False Sell Tap: False

Checking Pivot: R2 Level: 4532.5125
Body ratio: 0.537
Buy Tap: False Sell Tap: False

Checking Pivot: R3 Level: 4600.7297
Body ratio: 0.537
Buy Tap: False Sell Tap: False

Checking Pivot: S1 Level: 4353.9335
Body ratio: 0.537
Buy Tap: False Sell Tap: False

Checking Pivot: S2 Level: 4311.7888
Body ratio: 0.537
Buy Tap: False Sell Tap: False

Checking Pivot: S3 Level: 4243.5717
Body ratio: 0.431
Buy Tap: False Sell Tap: False

Checking Pivot: P Level: 4422.1507
Body ratio: 0.431
Buy Tap: False Sell Tap: False

Checking Pivot: R1 Level: 4490.3678
Body ratio: 0.4

Body ratio: 0.636
Buy Tap: False Sell Tap: False

Checking Pivot: R1 Level: 4490.3678
Body ratio: 0.636
Buy Tap: False Sell Tap: False

Checking Pivot: R2 Level: 4532.5125
Body ratio: 0.636
Buy Tap: False Sell Tap: False

Checking Pivot: R3 Level: 4600.7297
Body ratio: 0.636
Buy Tap: False Sell Tap: False

Checking Pivot: S1 Level: 4353.9335
Body ratio: 0.636
Buy Tap: False Sell Tap: False

Checking Pivot: S2 Level: 4311.7888
Body ratio: 0.636
Buy Tap: False Sell Tap: False

Checking Pivot: S3 Level: 4243.5717
Body ratio: 0.646
Buy Tap: False Sell Tap: False

Checking Pivot: P Level: 4422.1507
Body ratio: 0.646
Buy Tap: False Sell Tap: False

Checking Pivot: R1 Level: 4490.3678
Body ratio: 0.646
Buy Tap: False Sell Tap: False

Checking Pivot: R2 Level: 4532.5125
Body ratio: 0.646
Buy Tap: False Sell Tap: False

Checking Pivot: R3 Level: 4600.7297
Body ratio: 0.646
Buy Tap: False Sell Tap: False

Checking Pivot: S1 Level: 4353.9335
Body ratio: 0.646
Buy Tap: False Sell Tap: False

Check


Checking Pivot: R3 Level: 4668.134
Body ratio: 0.457
Buy Tap: False Sell Tap: False

Checking Pivot: S1 Level: 4465.3739
Body ratio: 0.457
Buy Tap: False Sell Tap: False

Checking Pivot: S2 Level: 4430.7491
Body ratio: 0.457
Buy Tap: False Sell Tap: False

Checking Pivot: S3 Level: 4374.704
Body ratio: 0.677
Buy Tap: False Sell Tap: False

Checking Pivot: P Level: 4521.419
Body ratio: 0.677
Buy Tap: False Sell Tap: False

Checking Pivot: R1 Level: 4577.4641
Body ratio: 0.677
Buy Tap: False Sell Tap: False

Checking Pivot: R2 Level: 4612.0889
Body ratio: 0.677
Buy Tap: False Sell Tap: False

Checking Pivot: R3 Level: 4668.134
Body ratio: 0.677
Buy Tap: False Sell Tap: False

Checking Pivot: S1 Level: 4465.3739
Body ratio: 0.677
Buy Tap: False Sell Tap: False

Checking Pivot: S2 Level: 4430.7491
Body ratio: 0.677
Buy Tap: False Sell Tap: False

Checking Pivot: S3 Level: 4374.704
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time filter
Blocked: Trading time 

Body ratio: 0.107
Rejected: Weak candle

Checking Pivot: R1 Level: 4577.4641
Body ratio: 0.107
Rejected: Weak candle

Checking Pivot: R2 Level: 4612.0889
Body ratio: 0.107
Rejected: Weak candle

Checking Pivot: R3 Level: 4668.134
Body ratio: 0.107
Rejected: Weak candle

Checking Pivot: S1 Level: 4465.3739
Body ratio: 0.107
Rejected: Weak candle

Checking Pivot: S2 Level: 4430.7491
Body ratio: 0.107
Rejected: Weak candle

Checking Pivot: S3 Level: 4374.704
Body ratio: 0.725
Buy Tap: False Sell Tap: False

Checking Pivot: P Level: 4521.419
Body ratio: 0.725
Buy Tap: False Sell Tap: False

Checking Pivot: R1 Level: 4577.4641
Body ratio: 0.725
Buy Tap: False Sell Tap: False

Checking Pivot: R2 Level: 4612.0889
Body ratio: 0.725
Buy Tap: False Sell Tap: False

Checking Pivot: R3 Level: 4668.134
Body ratio: 0.725
Buy Tap: False Sell Tap: False

Checking Pivot: S1 Level: 4465.3739
Body ratio: 0.725
Buy Tap: True Sell Tap: False

Checking Pivot: S2 Level: 4430.7491
Body ratio: 0.725
Buy Tap: F

IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

